# Part 1 — Does the LoRA+ learning-rate claim transfer to a vision–language model?

**Paper:** LoRA+: Efficient Low Rank Adaptation of Large Models (Hayou, Ghosh & Yu, 2024).

**Central claim.** For wide models, standard LoRA, which uses one learning rate for both adapter matrices, does not
learn features efficiently. Setting $\eta_B = \lambda\,\eta_A$ with $\lambda \gg 1$ (the paper recommends ≈ $2^4$) fixes
this and improves both final performance and convergence speed (the paper reports up to ~2× faster fine-tuning).
The key evidence is the paper's $(\eta_A, \eta_B)$ grid: the best cell lies **off the diagonal**, at $\eta_B > \eta_A$.

**Translated hypotheses (SmolVLM-256M, VQAv2).**
- **H1 (location of the optimum):** in an $(\eta_A, \lambda)$ grid, the best validation accuracy occurs at $\lambda > 1$, and
  the best LoRA+ cell beats the best $\lambda = 1$ cell (LoRA with a tuned LR).
- **H2a (final accuracy):** at a fixed training budget, tuned LoRA+ has higher test VQA accuracy than tuned LoRA (3 seeds).
- **H2b (speed):** LoRA+ reaches a target validation accuracy / loss in fewer steps. *H2a and H2b are separate claims
  and are measured separately.*
- **H3 (mechanism):** with $\lambda = 1$, the learned update $\Delta W = B_tA_t$ comes almost entirely from $B$ moving
  over nearly frozen random features $A_0$; LoRA+ increases the share that comes from $A$ actually learning.
- **Bonus (vision-specific):** is the benefit carried by the vision encoder or the language decoder? Tested by applying
  the ratio to only one tower.

> Check the exact grid ranges, recommended λ and speed-up numbers against the paper (Figs. on RoBERTa / GPT-2 / Llama, §4–5)
> before quoting them in the report.

In [ ]:
import os; os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip -q install peft datasets accelerate scipy
import transformers, peft, torch
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Config

In [ ]:
import os, re, gc, json, math, time, random, itertools
import numpy as np, pandas as pd
import torch.nn as nn
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor, TrainingArguments, Trainer, TrainerCallback, set_seed
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:
    from transformers import AutoModelForVision2Seq as AutoVLM
from peft import LoraConfig, get_peft_model

DATA_SEED = 42                     # fixes the data split for every run
SEEDS     = [42, 43, 44]           # training seeds for the main comparison
MODEL_ID  = "HuggingFaceTB/SmolVLM-256M-Instruct"
DATASET   = "merve/vqav2-small"    # VQAv2 validation subset (parquet)
N_TRAIN, N_VAL, N_TEST = 3000, 300, 500
BATCH     = 8
RANK, ALPHA, DROPOUT = 8, 16, 0.05
PROMPT    = "{q}\nAnswer with a single word or short phrase."
# LoRA on every attention + MLP linear layer of both towers (vision encoder and language decoder)
TARGET_REGEX = r".*(vision_model|text_model).*\.(q_proj|k_proj|v_proj|o_proj|out_proj|fc1|fc2|gate_proj|up_proj|down_proj)"

# (eta_A, lambda) grid — short runs, selected on the validation split. lambda = 1 is standard LoRA (an LR sweep).
GRID_LR_A   = [5e-5, 1e-4, 2e-4, 5e-4]
GRID_RATIO  = [1, 4, 16, 32]
GRID_EPOCHS, GRID_N_TRAIN = 1, 1500
MAIN_EPOCHS = 2
N_PROBES    = 8                    # checkpoints at which val accuracy + A/B shares are measured in main runs

QUICK = False                      # True = tiny smoke test of the whole pipeline (~minutes)
if QUICK:
    N_TRAIN, N_VAL, N_TEST, GRID_N_TRAIN = 200, 64, 64, 200
    GRID_LR_A, GRID_RATIO, SEEDS, MAIN_EPOCHS = [2e-4], [1, 16], [42], 1

random.seed(DATA_SEED); np.random.seed(DATA_SEED); set_seed(DATA_SEED)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/EN4554_A01/part1_loraplus" + ("_quick" if QUICK else "")
except ImportError:
    OUT_DIR = "./outputs/part1_loraplus" + ("_quick" if QUICK else "")
os.makedirs(OUT_DIR, exist_ok=True); print("Outputs ->", OUT_DIR)

## 2. Data — VQAv2 subset, split **by image** into train / val (model selection) / test (final report)

In [ ]:
raw = load_dataset(DATASET, split="validation")
print(raw)

rng = random.Random(DATA_SEED)
if "image_id" in raw.column_names:
    img_ids = raw["image_id"]
    uniq = sorted(set(img_ids)); rng.shuffle(uniq)
    n10 = len(uniq) // 10
    val_imgs, test_imgs = set(uniq[:n10]), set(uniq[n10:2 * n10])
    split_of = lambda x: "val" if x in val_imgs else "test" if x in test_imgs else "train"
    buckets = {"train": [], "val": [], "test": []}
    for i, x in enumerate(img_ids):
        buckets[split_of(x)].append(i)
else:
    idx = list(range(len(raw))); rng.shuffle(idx); n10 = len(idx) // 10
    buckets = {"val": idx[:n10], "test": idx[n10:2 * n10], "train": idx[2 * n10:]}
for v in buckets.values():
    rng.shuffle(v)
train_ds = raw.select(buckets["train"][:N_TRAIN])
val_ds   = raw.select(buckets["val"][:N_VAL])
test_ds  = raw.select(buckets["test"][:N_TEST])
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}  ({DATASET}/validation, data seed {DATA_SEED})")

def gt_answers(e):
    a = e.get("answers")
    if a:
        return [x["answer"] if isinstance(x, dict) else x for x in a]
    return [e["multiple_choice_answer"]]

ex = train_ds[0]; print(ex["question"], "->", ex["multiple_choice_answer"], "| all:", gt_answers(ex)[:10])

## 3. Processor and assistant-only-loss collator

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.do_image_splitting = False    # one 512px tile -> 64 image tokens
tok = processor.tokenizer
ASSIST_IDS = tok.encode("Assistant:", add_special_tokens=False)

def build_msgs(q, a=None):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT.format(q=q)}]}]
    if a is not None:
        msgs.append({"role": "assistant", "content": [{"type": "text", "text": a}]})
    return msgs

def find_last(seq, sub):
    for i in range(len(seq) - len(sub), -1, -1):
        if seq[i:i + len(sub)] == sub:
            return i
    return -1

def train_collate(examples):
    texts  = [processor.apply_chat_template(build_msgs(e["question"], e["multiple_choice_answer"])) for e in examples]
    images = [[e["image"].convert("RGB")] for e in examples]
    tok.padding_side = "right"
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    for i, ids in enumerate(batch["input_ids"].tolist()):
        j = find_last(ids, ASSIST_IDS)
        labels[i, : (j + len(ASSIST_IDS)) if j >= 0 else len(ids)] = -100   # loss on the assistant turn only
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

b = train_collate([train_ds[0], train_ds[1]])
for i in range(2):
    print("supervised:", repr(tok.decode(b["labels"][i][b["labels"][i] != -100])), "| seq len", b["input_ids"].shape[1])

## 4. VQA evaluation (VQAv2 soft accuracy: min(#annotators agreeing / 3, 1))

In [ ]:
def normalize(s):
    s = s.lower().strip().split("\n")[0]
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

@torch.no_grad()
def evaluate(model, ds, bs=32, max_new_tokens=10):
    was_training = model.training
    model.eval(); tok.padding_side = "left"
    rows = []
    for s in range(0, len(ds), bs):
        exs = ds.select(range(s, min(s + bs, len(ds))))
        texts  = [processor.apply_chat_template(build_msgs(e["question"]), add_generation_prompt=True) for e in exs]
        images = [[e["image"].convert("RGB")] for e in exs]
        inputs = processor(text=texts, images=images, return_tensors="pt", padding=True).to(model.device)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gens = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for e, g in zip(exs, gens):
            p = normalize(g); gts = [normalize(a) for a in gt_answers(e)]
            rows.append(dict(question=e["question"], pred=p, gt=e["multiple_choice_answer"],
                             acc=min(sum(a == p for a in gts) / 3, 1.0)))
    if was_training:
        model.train()
    return float(np.mean([r["acc"] for r in rows])), rows

def load_model():
    return AutoVLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).float().to("cuda")  # fp32 master weights + fp16 autocast

## 5. LoRA+ optimiser and probes

LoRA+ is just two AdamW parameter groups: `lora_B` gets $\lambda\,\eta_A$, everything else gets $\eta_A$. The Trainer's
linear schedule scales every group by the same factor, so the ratio is kept throughout training. `ratio_vision` /
`ratio_text` override λ for one tower (used in the bonus).

**A/B share probe (H3).** With $A_0$ the initial (random) A: $B_tA_t = \underbrace{B_tA_0}_{\text{B learns on random features}} + \underbrace{B_t(A_t-A_0)}_{\text{A learns features}}$.
We log $\text{share}_A = \|B_t(A_t-A_0)\|_F / (\|B_tA_0\|_F + \|B_t(A_t-A_0)\|_F)$ and the relative movement $\|A_t-A_0\|_F/\|A_0\|_F$, averaged per tower.

In [ ]:
def make_optimizer(model, lr_A, ratio, ratio_vision=None, ratio_text=None):
    groups = {}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        r = 1.0
        if "lora_B" in n:
            r = ratio
            if ratio_vision is not None and "vision_model" in n: r = ratio_vision
            if ratio_text is not None and "text_model" in n: r = ratio_text
        groups.setdefault(float(r), []).append(p)
    print("  param groups:", {f"lr={lr_A*r:.1e}": sum(p.numel() for p in ps) for r, ps in groups.items()})
    return torch.optim.AdamW([{"params": ps, "lr": lr_A * r} for r, ps in groups.items()], lr=lr_A, weight_decay=0.0)

def lora_modules(model):
    return {n.replace("base_model.model.", ""): m for n, m in model.named_modules()
            if hasattr(m, "lora_A") and "default" in getattr(m, "lora_A", {})}

class Probe(TrainerCallback):
    # At chosen steps: VQA accuracy on the val split (H2b) and A/B update shares per tower (H3).
    def __init__(self, model, steps, eval_acc):
        self.model, self.steps, self.eval_acc = model, set(steps), eval_acc
        self.mods = lora_modules(model)
        self.A0 = {n: m.lora_A["default"].weight.detach().float().clone() for n, m in self.mods.items()}
        self.acc_curve, self.shares = [], []

    @torch.no_grad()
    def measure_shares(self, step):
        agg = {}
        for n, m in self.mods.items():
            A, B = m.lora_A["default"].weight.float(), m.lora_B["default"].weight.float()
            dA = A - self.A0[n]
            fromB, fromA = (B @ self.A0[n]).norm().item(), (B @ dA).norm().item()
            tw = "vision" if "vision_model" in n else "text"
            agg.setdefault(tw, []).append((fromA / (fromA + fromB + 1e-12), (dA.norm() / self.A0[n].norm()).item()))
        for tw, v in agg.items():
            v = np.array(v)
            self.shares.append(dict(step=step, tower=tw, share_A=v[:, 0].mean(), rel_move_A=v[:, 1].mean()))

    def on_step_end(self, args, state, control, **kw):
        if state.global_step in self.steps:
            self.measure_shares(state.global_step)
            if self.eval_acc:
                acc, _ = evaluate(self.model, val_ds)
                self.acc_curve.append((state.global_step, acc))
                print(f"  step {state.global_step}: val acc {100*acc:.2f}%")

## 6. One run = (η_A, λ, seed, budget). Results are cached on Drive, so reruns resume.

In [ ]:
def run(name, lr_A, ratio, seed, epochs, n_train, probe_acc=False, ratio_vision=None, ratio_text=None, eval_test=False):
    path = f"{OUT_DIR}/runs/{name}"; res_file = f"{path}/results.json"
    if os.path.exists(res_file):
        return json.load(open(res_file))
    os.makedirs(path, exist_ok=True); set_seed(seed)
    print(f"=== {name}: eta_A={lr_A:g} lambda={ratio} seed={seed} epochs={epochs} n={n_train}")

    model = get_peft_model(load_model(), LoraConfig(r=RANK, lora_alpha=ALPHA, lora_dropout=DROPOUT,
                                                    target_modules=TARGET_REGEX, bias="none"))
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_tot = sum(p.numel() for p in model.parameters())
    tr_ds = train_ds.select(range(n_train))
    total_steps = math.ceil(n_train / BATCH) * epochs
    probe_steps = sorted(set(np.linspace(total_steps / N_PROBES, total_steps, N_PROBES).round().astype(int).tolist()))
    probe = Probe(model, probe_steps, eval_acc=probe_acc)

    args = TrainingArguments(
        output_dir=f"{path}/tmp", num_train_epochs=epochs, per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=16, learning_rate=lr_A, lr_scheduler_type="linear",
        warmup_steps=int(0.05 * total_steps), max_grad_norm=1.0, fp16=True,
        logging_steps=10, eval_strategy="steps", eval_steps=max(10, total_steps // 20), prediction_loss_only=True,
        save_strategy="no", report_to="none", remove_unused_columns=False, label_names=["labels"],
        dataloader_num_workers=2, seed=seed)
    opt = make_optimizer(model, lr_A, ratio, ratio_vision, ratio_text)
    trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=val_ds,
                      data_collator=train_collate, callbacks=[probe], optimizers=(opt, None))

    t0 = time.time(); trainer.train(); train_min = (time.time() - t0) / 60
    lh = trainer.state.log_history
    train_loss = [(x["step"], x["loss"]) for x in lh if "loss" in x]
    val_loss   = [(x["step"], x["eval_loss"]) for x in lh if "eval_loss" in x]
    grad_norm  = [x["grad_norm"] for x in lh if "grad_norm" in x]
    val_acc = probe.acc_curve[-1][1] if probe.acc_curve else evaluate(model, val_ds)[0]
    res = dict(name=name, lr_A=lr_A, ratio=ratio, ratio_vision=ratio_vision, ratio_text=ratio_text, seed=seed,
               epochs=epochs, n_train=n_train, steps=total_steps, trainable=n_tr, total=n_tot, train_min=train_min,
               train_loss=train_loss, val_loss=val_loss, max_grad_norm=float(np.nanmax(grad_norm)) if grad_norm else None,
               val_acc=val_acc, acc_curve=probe.acc_curve, shares=probe.shares,
               diverged=bool(not train_loss or not np.isfinite(train_loss[-1][1])))
    if eval_test:
        res["test_acc"], res["test_preds"] = evaluate(model, test_ds)
    print(f"  -> val acc {100*val_acc:.2f}%" + (f" | test acc {100*res['test_acc']:.2f}%" if eval_test else "")
          + f" | {train_min:.1f} min")
    json.dump(res, open(res_file, "w"))
    del model, trainer, probe, opt; gc.collect(); torch.cuda.empty_cache()
    return res

### 6a. Baseline: the untouched model (val and test)

In [ ]:
base_file = f"{OUT_DIR}/baseline.json"
if os.path.exists(base_file):
    baseline = json.load(open(base_file))
else:
    m = load_model()
    baseline = dict(val_acc=evaluate(m, val_ds)[0]); baseline["test_acc"], baseline["test_preds"] = evaluate(m, test_ds)
    json.dump(baseline, open(base_file, "w")); del m; gc.collect(); torch.cuda.empty_cache()
print(f"baseline: val {100*baseline['val_acc']:.2f}% | test {100*baseline['test_acc']:.2f}%")

### 6b. (η_A, λ) grid — H1

Short runs (`GRID_N_TRAIN` examples, `GRID_EPOCHS` epoch, seed 42). The λ = 1 column is standard LoRA swept over learning
rates, so "LoRA+ only wins because B has a larger LR" is ruled out if the best cell has λ > 1 and beats every λ = 1 cell.
**Time the first run** and shrink the grid if the session budget is tight.

In [ ]:
grid = [run(f"grid_lr{lr:g}_r{r}", lr, r, SEEDS[0], GRID_EPOCHS, GRID_N_TRAIN)
        for lr, r in itertools.product(GRID_LR_A, GRID_RATIO)]
gdf = pd.DataFrame([dict(lr_A=g["lr_A"], ratio=g["ratio"], lr_B=g["lr_A"] * g["ratio"], val_acc=100 * g["val_acc"],
                         final_val_loss=g["val_loss"][-1][1] if g["val_loss"] else np.nan, diverged=g["diverged"],
                         max_grad_norm=g["max_grad_norm"]) for g in grid])
gdf.to_csv(f"{OUT_DIR}/grid.csv", index=False)
display(gdf.round(4))

best_lora = gdf[gdf.ratio == 1].sort_values("val_acc", ascending=False).iloc[0]
best_plus = gdf[gdf.ratio > 1].sort_values("val_acc", ascending=False).iloc[0]
print(f"best LoRA  (lambda=1): eta={best_lora.lr_A:g}  val acc {best_lora.val_acc:.2f}%")
print(f"best LoRA+ (lambda>1): eta_A={best_plus.lr_A:g} lambda={best_plus.ratio:g}  val acc {best_plus.val_acc:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col, title, cmap in [(axes[0], "val_acc", "val VQA acc (%)", "viridis"),
                             (axes[1], "final_val_loss", "final val loss", "viridis_r")]:
    piv = gdf.pivot(index="lr_A", columns="ratio", values=col)
    im = ax.imshow(piv.values, cmap=cmap, aspect="auto", origin="lower")
    ax.set_xticks(range(len(piv.columns)), [f"λ={c:g}" for c in piv.columns])
    ax.set_yticks(range(len(piv.index)), [f"{i:g}" for i in piv.index]); ax.set_ylabel("η_A")
    for (i, j), v in np.ndenumerate(piv.values):
        ax.text(j, i, f"{v:.2f}" if col == "val_acc" else f"{v:.3f}", ha="center", va="center", color="w", fontsize=9)
    ax.set_title(title); fig.colorbar(im, ax=ax)
plt.suptitle("H1: where is the optimum in the (η_A, λ = η_B/η_A) grid?"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/grid_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

### 6c. Main comparison — H2a (final accuracy) and H2b (speed)

Full budget (`N_TRAIN`, `MAIN_EPOCHS`), 3 seeds, val accuracy probed during training, test accuracy at the end.
- **LoRA (tuned):** best η from the λ = 1 column.
- **LoRA+ (tuned):** best (η_A, λ) cell with λ > 1.
- **LoRA+ (drop-in, λ = 16):** LoRA's tuned η used as η_A, with the paper's default ratio, to check whether the recipe works without retuning.

In [ ]:
configs = {
    "LoRA (tuned)":      (float(best_lora.lr_A), 1),
    "LoRA+ (tuned)":     (float(best_plus.lr_A), float(best_plus.ratio)),
    "LoRA+ (drop-in 16)": (float(best_lora.lr_A), 16),
}
main = {k: [run(f"main_{k.split()[0]}_{k.split()[1].strip('()')}_lr{lr:g}_r{r:g}_s{s}".replace("+", "plus"),
                lr, r, s, MAIN_EPOCHS, N_TRAIN, probe_acc=True, eval_test=True) for s in SEEDS]
        for k, (lr, r) in configs.items()}

## 7. Results

In [ ]:
lora_runs = main["LoRA (tuned)"]
target_acc  = 0.95 * np.mean([r["acc_curve"][-1][1] for r in lora_runs])   # 95% of LoRA's final val acc
target_loss = np.mean([r["val_loss"][-1][1] for r in lora_runs])           # LoRA's final val loss

def first_step(curve, ok):
    return next((s for s, v in curve if ok(v)), np.nan)

rows = [dict(method="Baseline (zero-shot)", test_acc=100 * baseline["test_acc"])]
for k, runs in main.items():
    ta = [100 * r["test_acc"] for r in runs]
    s_acc  = [first_step(r["acc_curve"], lambda v: v >= target_acc) for r in runs]
    s_loss = [first_step(r["val_loss"], lambda v: v <= target_loss) for r in runs]
    rows.append(dict(method=k, eta_A=runs[0]["lr_A"], ratio=runs[0]["ratio"],
                     test_acc=np.mean(ta), test_acc_std=np.std(ta),
                     final_train_loss=np.mean([r["train_loss"][-1][1] for r in runs]),
                     final_val_loss=np.mean([r["val_loss"][-1][1] for r in runs]),
                     steps_to_95pct_LoRA_acc=np.nanmean(s_acc), steps_to_LoRA_val_loss=np.nanmean(s_loss),
                     max_grad_norm=np.max([r["max_grad_norm"] for r in runs]),
                     trainable_pct=100 * runs[0]["trainable"] / runs[0]["total"], train_min=np.mean([r["train_min"] for r in runs])))
res_tbl = pd.DataFrame(rows)
lo = res_tbl.set_index("method").loc["LoRA (tuned)"]
res_tbl["speedup_vs_LoRA (loss)"] = lo.steps_to_LoRA_val_loss / res_tbl.steps_to_LoRA_val_loss
res_tbl.to_csv(f"{OUT_DIR}/main_table.csv", index=False)
print(f"target val acc = {100*target_acc:.2f}%   target val loss = {target_loss:.4f}   (seeds {SEEDS})")
display(res_tbl.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for k, runs in main.items():
    # val accuracy curve (H2b)
    steps = [s for s, _ in runs[0]["acc_curve"]]
    A = np.array([[v for _, v in r["acc_curve"]] for r in runs]) * 100
    x = [0] + steps; mu = np.r_[100 * baseline["val_acc"], A.mean(0)]; sd = np.r_[0, A.std(0)]
    axes[0].plot(x, mu, marker="o", label=k); axes[0].fill_between(x, mu - sd, mu + sd, alpha=.2)
    # val loss curve (H2b)
    L = np.array([[v for _, v in r["val_loss"]] for r in runs]); ls = [s for s, _ in runs[0]["val_loss"]]
    axes[1].plot(ls, L.mean(0), label=k); axes[1].fill_between(ls, L.mean(0) - L.std(0), L.mean(0) + L.std(0), alpha=.2)
    # train loss (stability)
    T = np.array([[v for _, v in r["train_loss"]] for r in runs]); ts = [s for s, _ in runs[0]["train_loss"]]
    axes[2].plot(ts, T.mean(0), label=k, lw=1)
axes[0].axhline(100 * target_acc, ls="--", c="gray", lw=1, label="target (95% of LoRA)")
axes[1].axhline(target_loss, ls="--", c="gray", lw=1, label="LoRA final val loss")
for ax, t in zip(axes, ["val VQA acc (%)", "val loss", "train loss"]):
    ax.set_title(t); ax.set_xlabel("step"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/curves.png", dpi=150, bbox_inches="tight"); plt.show()

### 7a. Mechanism (H3): how much of ΔW comes from A actually learning?

In [ ]:
sh = pd.concat([pd.DataFrame(r["shares"]).assign(method=k, seed=r["seed"]) for k, runs in main.items() for r in runs])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (k, tw), g in sh.groupby(["method", "tower"]):
    g = g.groupby("step")[["share_A", "rel_move_A"]].mean()
    ls = "-" if tw == "vision" else "--"
    axes[0].plot(g.index, g.share_A, ls, marker=".", label=f"{k} / {tw}")
    axes[1].plot(g.index, g.rel_move_A, ls, marker=".", label=f"{k} / {tw}")
axes[0].set_title("share of ΔW from A learning:  ‖B(A−A₀)‖ / (‖BA₀‖+‖B(A−A₀)‖)")
axes[1].set_title("relative movement of A:  ‖A−A₀‖ / ‖A₀‖"); axes[1].set_yscale("log")
for ax in axes: ax.set_xlabel("step"); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/ab_shares.png", dpi=150, bbox_inches="tight"); plt.show()
display(sh[sh.step == sh.step.max()].groupby(["method", "tower"])[["share_A", "rel_move_A"]].mean().round(4))

### 7b. Bonus (vision-specific): which tower carries the LoRA+ benefit?

Same budget and seed as the main runs, with the tuned LoRA+ ratio applied to the B matrices of **only** the vision encoder
or **only** the language decoder (the other tower keeps λ = 1). If most of the gain comes from the vision tower, the
claim's benefit in a VLM sits mainly in visual feature learning, which an NLP-only paper cannot show.

In [ ]:
lr, lam = configs["LoRA+ (tuned)"]
bonus = {
    "λ only in vision encoder":   run(f"bonus_vision_lr{lr:g}_r{lam:g}", lr, 1, SEEDS[0], MAIN_EPOCHS, N_TRAIN,
                                     probe_acc=True, eval_test=True, ratio_vision=lam, ratio_text=1),
    "λ only in language decoder": run(f"bonus_text_lr{lr:g}_r{lam:g}", lr, 1, SEEDS[0], MAIN_EPOCHS, N_TRAIN,
                                     probe_acc=True, eval_test=True, ratio_vision=1, ratio_text=lam),
}
ref = {"LoRA (tuned)": main["LoRA (tuned)"][0], "LoRA+ (both towers)": main["LoRA+ (tuned)"][0]}
btbl = pd.DataFrame([dict(setting=k, test_acc=100 * r["test_acc"], final_val_loss=r["val_loss"][-1][1],
                          steps_to_LoRA_val_loss=first_step(r["val_loss"], lambda v: v <= target_loss))
                     for k, r in {**ref, **bonus}.items()])
btbl.to_csv(f"{OUT_DIR}/bonus_table.csv", index=False); display(btbl.round(4))

## 8. Reading the result

| Claim | Supported if… |
|---|---|
| **H1** optimum off-diagonal | best grid cell has λ > 1 **and** beats the best λ = 1 cell (tuned LoRA) |
| **H2a** better final accuracy | LoRA+ (tuned) test acc − LoRA (tuned) test acc is larger than the seed std |
| **H2b** faster convergence | LoRA+ reaches the target val loss / acc in fewer steps (`speedup` > 1) |
| **H3** mechanism | LoRA's `share_A` is near 0 and LoRA+ raises it |

H2a and H2b can come apart. For example, LoRA+ may get there faster but plateau at the same accuracy; report each
on its own. Selection used the **val** split only; every number in the main table is on the **test** split.
For reproducibility, report: data seed 42, training seeds 42/43/44, split sizes from §2, rank 8, α 16, and the grid.